# TaxaLens v0.8 — Whole-Image Baseline 🪰
One complete image → one DINOv3 embedding. No anatomy masks, no 2×2 tiles in the default experiment.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!git clone https://github.com/SaniyaSani/TaxaLens.git || true
%cd TaxaLens
!pip -q install -r requirements-foundation.txt


## 1. Check source files
Edit `configs/sources_wholefly_v08.example.json` paths if your data live on Drive.


In [ ]:
!python scripts/source_status_v08.py --config configs/sources_wholefly_v08.example.json


## 2. Cache DINOv3
Small is the default Colab/T4 backbone.


In [ ]:
!python scripts/cache_backbone.py --model facebook/dinov3-vits16-pretrain-lvd1689m


## 3. Build / verify master manifest
If your normalized source manifests are ready, assemble and deduplicate them.


In [ ]:
!python scripts/prepare_corpus.py --config configs/sources_wholefly_v08.example.json --stage assemble


## 4. Plan the 100k whole-fly corpus


In [ ]:
MASTER='data/corpus_v08/master_manifest.parquet'
!python scripts/run_wholefly_v08.py --master-manifest $MASTER --stage plan


## 5. Embed whole images in resumable shards
Default config uses `tile_grid=1`, so every photo stays whole. Finished shards are skipped.


In [ ]:
!python scripts/run_wholefly_v08.py --master-manifest $MASTER --stage embed --max-shards 2


## 6. Train after all shards are complete


In [ ]:
!python scripts/run_wholefly_v08.py --master-manifest $MASTER --stage train


The training report includes overall metrics, per-source metrics, BIOSCAN source-split metrics where available, and top confusion pairs. Use those errors to decide whether a targeted head/wing/morphology module is worth building.
